# Extending the World Cup prediction model

This notebook shows how to create a first reproducible covariate: **recent form**.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

RESULTS_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
ROOT = Path("..")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

PREDICTION_DATE = pd.Timestamp("2026-06-11")
N_RECENT_MATCHES = 10


## Load historical results

Only use matches before the prediction date to avoid leakage.

In [ ]:
results = pd.read_csv(RESULTS_URL)
results["date"] = pd.to_datetime(results["date"])

results = results.dropna(
    subset=["date", "home_team", "away_team", "home_score", "away_score"]
).copy()

results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)

historical = results[results["date"] < PREDICTION_DATE].copy()
historical = historical.sort_values("date").reset_index(drop=True)

historical.tail()


## Convert match results to team-level rows

In [ ]:
def match_rows_by_team(matches):
    rows = []
    for _, row in matches.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        home_goals = int(row["home_score"])
        away_goals = int(row["away_score"])

        if home_goals > away_goals:
            home_points, away_points = 3, 0
        elif home_goals < away_goals:
            home_points, away_points = 0, 3
        else:
            home_points, away_points = 1, 1

        rows.append({
            "date": row["date"],
            "team": home,
            "opponent": away,
            "goals_for": home_goals,
            "goals_against": away_goals,
            "goal_diff": home_goals - away_goals,
            "points": home_points,
            "tournament": row["tournament"],
        })

        rows.append({
            "date": row["date"],
            "team": away,
            "opponent": home,
            "goals_for": away_goals,
            "goals_against": home_goals,
            "goal_diff": away_goals - home_goals,
            "points": away_points,
            "tournament": row["tournament"],
        })
    return pd.DataFrame(rows)

team_match_rows = match_rows_by_team(historical)
team_match_rows.head()


## Compute recent-form covariates

In [ ]:
def compute_recent_form(team_match_rows, n_recent=10):
    rows = []
    for team, group in team_match_rows.groupby("team"):
        recent = group.sort_values("date").tail(n_recent)
        if recent.empty:
            continue
        rows.append({
            "team": team,
            "recent_matches": len(recent),
            "recent_form_points": recent["points"].mean(),
            "recent_goal_diff": recent["goal_diff"].mean(),
            "recent_goals_for": recent["goals_for"].mean(),
            "recent_goals_against": recent["goals_against"].mean(),
        })
    return pd.DataFrame(rows).sort_values("team").reset_index(drop=True)

recent_form = compute_recent_form(team_match_rows, n_recent=N_RECENT_MATCHES)
recent_form.head()


## Save starter covariate file

In [ ]:
covariates = recent_form.copy()
covariates["fifa_rank"] = np.nan
covariates["fifa_points"] = np.nan
covariates["market_value_million_eur"] = np.nan
covariates["injury_index"] = 0.0

DATA_DIR.mkdir(exist_ok=True)
output_path = DATA_DIR / "team_covariates_2026.csv"
covariates.to_csv(output_path, index=False)
output_path


## Example covariate gaps

Most covariates should enter the model as `team value - opponent value`.

In [ ]:
def get_covariate(team, covariate_dict, name, default=0.0):
    if team not in covariate_dict:
        return default
    value = covariate_dict[team].get(name, default)
    if pd.isna(value):
        return default
    return float(value)

covariate_dict = covariates.set_index("team").to_dict(orient="index")

team = "France"
opponent = "Brazil"

recent_form_gap = (
    get_covariate(team, covariate_dict, "recent_form_points")
    - get_covariate(opponent, covariate_dict, "recent_form_points")
)

recent_form_gap


## Starter expected-goals function with covariates

The coefficients are placeholders. A stronger version estimates them using `estimate_elo_goals_relation.py`.

In [ ]:
GOALS_INTERCEPT = 0.17
GOALS_ELO_COEF = 0.12
GOALS_HOST_COEF = 0.17
GOALS_RECENT_FORM_COEF = 0.05
GOALS_MARKET_VALUE_COEF = 0.04
GOALS_INJURY_COEF = -0.10

ELO_GAP_SHRINKAGE = 1.00
MAX_ELO_GAP_PER_100 = 2.5
HOSTS = {"United States", "Mexico", "Canada"}

def expected_goals_with_covariates(team, opponent, elo, covariates):
    team_rating = elo.get(team, 1500)
    opponent_rating = elo.get(opponent, 1500)

    rating_gap = (team_rating - opponent_rating) / 100
    rating_gap = np.clip(rating_gap, -MAX_ELO_GAP_PER_100, MAX_ELO_GAP_PER_100)
    rating_gap = ELO_GAP_SHRINKAGE * rating_gap

    host_boost = 1 if team in HOSTS else 0

    recent_form_gap = (
        get_covariate(team, covariates, "recent_form_points")
        - get_covariate(opponent, covariates, "recent_form_points")
    )

    market_value_gap = (
        math.log1p(get_covariate(team, covariates, "market_value_million_eur"))
        - math.log1p(get_covariate(opponent, covariates, "market_value_million_eur"))
    )

    injury_gap = (
        get_covariate(team, covariates, "injury_index")
        - get_covariate(opponent, covariates, "injury_index")
    )

    log_mu = (
        GOALS_INTERCEPT
        + GOALS_ELO_COEF * rating_gap
        + GOALS_HOST_COEF * host_boost
        + GOALS_RECENT_FORM_COEF * recent_form_gap
        + GOALS_MARKET_VALUE_COEF * market_value_gap
        + GOALS_INJURY_COEF * injury_gap
    )

    mu = math.exp(log_mu)
    return float(np.clip(mu, 0.15, 4.5))


## Discussion prompts

1. Which covariates are available before a match?
2. Which might duplicate Elo?
3. Which might leak future information?
4. How would you validate whether the covariate improves the model?